# BF16 Colab validation
A100 80GB required. Read repository README and THIRD_PARTY_NOTICES before running. Qwen Image is research-only. Run cells in order. This notebook is generated from reviewed scripts; original evidence is separate. No Drive or token required.


In [ ]:
"""CPU-only helpers shared by generated notebooks. No network or GPU work."""
import hashlib
import importlib.metadata
import json
import pathlib
import threading
import time
import uuid
import zipfile

MODEL_PROFILES = {
    "qwen3_8": {"repo": "huihui-ai/Huihui-Qwen3.8-27B-abliterated", "revision": "739e3c5b89849f6c238ce1e5b70008612ae42cdd", "ram_gib": 64, "disk_gib": 80, "files": ("preflight.json", "qwen3_8_bf16.json", "qwen_checkpoint_audit.json", "runtime_versions.json")},
    "qwen_image_2_1": {"repo": "Qwen/Qwen-Image-2.1", "revision": "d26bb61231c349cf6b7896fa83353113880e1ba3", "ram_gib": 48, "disk_gib": 55, "files": ("preflight.json", "qwen_image_2_1_bf16.json", "qwen_image_2_1_bf16_2048.png", "runtime_versions.json")},
}

def arm_shutdown(state, release, seconds=1500, timer_factory=threading.Timer, clock=time.monotonic):
    """One timer/deadline per kernel lifecycle; rerunning cannot extend it."""
    now = clock()
    deadline = state.setdefault("deadline", now + seconds)
    old = state.get("timer")
    if old is not None:
        old.cancel()
    timer = timer_factory(max(0, deadline - now), release)
    timer.daemon = True
    state["timer"] = timer
    timer.start()
    return deadline

def shutdown_now(state, release):
    """Keep fallback timer armed if immediate runtime release raises."""
    release()
    if state.get("timer") is not None:
        state["timer"].cancel()


def new_report_dir(base, profile):
    path = pathlib.Path(base) / (profile + "-" + uuid.uuid4().hex)
    path.mkdir(parents=True, exist_ok=False)
    return path


def require_resources(profile, available_ram_gib, disk_gib):
    config = MODEL_PROFILES[profile]
    if available_ram_gib < config["ram_gib"] or disk_gib < config["disk_gib"]:
        raise RuntimeError(f"STOP: {profile} conservatively requires available RAM >= {config['ram_gib']} GiB and free disk >= {config['disk_gib']} GiB. No model download started.")


def observed_distribution(package):
    try:
        dist = importlib.metadata.distribution(package)
    except importlib.metadata.PackageNotFoundError:
        return None
    source = None
    try:
        direct = json.loads(dist.read_text("direct_url.json") or "null")
        # Publish only an allowlisted source, never arbitrary URLs/usernames/tokens.
        if isinstance(direct, dict) and direct.get("url") == "https://github.com/huggingface/diffusers.git":
            commit = direct.get("vcs_info", {}).get("commit_id")
            if isinstance(commit, str) and len(commit) == 40 and all(c in "0123456789abcdef" for c in commit):
                source = {"url": direct["url"], "commit": commit}
    except (ValueError, TypeError):
        pass
    return {"version": dist.version, "source": source}


def export_allowlisted(report_dir, profile):
    """Archive a snapshot of approved files only; reject symlinks/missing files."""
    root = pathlib.Path(report_dir)
    payload = {}
    for name in MODEL_PROFILES[profile]["files"]:
        path = root / name
        if path.is_symlink() or not path.is_file():
            raise ValueError(f"Missing or unsafe expected evidence file: {name}")
        payload[name] = path.read_bytes()
    # Manifest covers payload entries, excluding itself by definition.
    manifest = {name: hashlib.sha256(data).hexdigest() for name, data in payload.items()}
    manifest_bytes = (json.dumps(manifest, indent=2) + "\n").encode()
    archive = root.parent / (root.name + "_evidence.zip")
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as zipped:
        for name, data in payload.items():
            zipped.writestr(name, data)
        zipped.writestr("sha256_manifest.json", manifest_bytes)
    (root / "sha256_manifest.json").write_bytes(manifest_bytes)
    return archive, manifest

MODEL_PROFILE='qwen_image_2_1'
from google.colab import runtime
if "shutdown_state" not in globals(): shutdown_state={}
arm_shutdown(shutdown_state, runtime.unassign)
print("Single 25-minute lifecycle deadline armed; rerunning does not extend it.")


In [ ]:
%pip install -q transformers==5.22.1 accelerate==1.15.0 huggingface_hub==1.33.0 safetensors==0.8.0


If Colab asks to restart the session after installation, restart and rerun from the timer cell. Keep Colab supplied PyTorch. Validated environment was Python 3.13.16 / torch 2.14.1+cu130; other environments are not validated.


In [ ]:
# Requires runtime_helpers and MODEL_PROFILE from the notebook bootstrap.
import os, sys, json, time, pathlib, shutil, subprocess, platform, urllib.request
from datetime import datetime, timezone
import torch
import psutil
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
PROFILE = MODEL_PROFILES[MODEL_PROFILE]
REPORT_DIR = new_report_dir("/content/a100_validation", MODEL_PROFILE)
report = {"timestamp_utc": datetime.now(timezone.utc).isoformat(), "python": sys.version.split()[0], "torch": torch.__version__, "gpu": [], "ram_gib": round(psutil.virtual_memory().total / 2**30, 2), "available_ram_gib": psutil.virtual_memory().available / 2**30, "disk_free_gib": shutil.disk_usage("/content").free / 2**30, "profile": MODEL_PROFILE, "resource_thresholds": {"available_ram_gib": PROFILE["ram_gib"], "free_disk_gib": PROFILE["disk_gib"]}}
assert torch.cuda.is_available(), "STOP: no GPU. No model download permitted."
p = torch.cuda.get_device_properties(0)
report["gpu"].append({"name": p.name, "vram_gib": p.total_memory / 2**30, "capability": list(torch.cuda.get_device_capability(0)), "bf16_supported": torch.cuda.is_bf16_supported()})
assert "A100" in p.name and p.total_memory / 2**30 > 70 and torch.cuda.is_bf16_supported(), "STOP: A100 80GB BF16 required; 40GB is not validated."
require_resources(MODEL_PROFILE, report["available_ram_gib"], report["disk_free_gib"])
url = "https://huggingface.co/api/models/" + PROFILE["repo"] + "/revision/" + PROFILE["revision"]
with urllib.request.urlopen(url, timeout=30) as response:
    data = json.load(response)
assert data.get("sha") == PROFILE["revision"], "STOP: resolved model revision mismatch."
report["models"] = {PROFILE["repo"]: {"revision": data["sha"], "gated": data.get("gated"), "license": data.get("cardData", {}).get("license"), "safetensors": data.get("safetensors")}}
print(json.dumps(report, ensure_ascii=False, indent=2))
(REPORT_DIR / "preflight.json").write_text(json.dumps(report, ensure_ascii=False, indent=2))
print("PREFLIGHT_ONLY: no model has been loaded or tested. Evidence directory:", REPORT_DIR)


In [ ]:
# Non-commercial technical evaluation under the Qwen Research License.
# Official checkpoint; every learned component remains BF16 (no INT8/FP8/NVFP4).
import subprocess, sys, importlib, time, json, collections, hashlib, gc
if "p" in globals():
    del p
    gc.collect()
    torch.cuda.empty_cache()
print("GPU_ALLOCATED_BEFORE_IMAGE_GIB",round(torch.cuda.memory_allocated()/2**30,3))
assert torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory/2**30 > 70, "STOP: A100 80GB required before model download."
DIFFUSERS_COMMIT="84532a058702c74ae5d1dd6ceb05683fc5d5e768"
assert not any(name == "diffusers" or name.startswith("diffusers.") for name in sys.modules), "STOP: diffusers already imported; restart runtime and rerun from bootstrap to avoid stale modules."
subprocess.check_call([sys.executable,"-m","pip","install","-q","git+https://github.com/huggingface/diffusers.git@"+DIFFUSERS_COMMIT])
installed_diffusers = observed_distribution("diffusers")
assert installed_diffusers and installed_diffusers["source"] and installed_diffusers["source"]["commit"] == DIFFUSERS_COMMIT, "STOP: installed Diffusers source commit does not match expected pin."
from diffusers import QwenImage21Pipeline
import diffusers, numpy as np
MODEL_ID=PROFILE["repo"]
REVISION=PROFILE["revision"]
assert torch.cuda.is_available() and "A100" in torch.cuda.get_device_name(0)
assert torch.cuda.memory_allocated()/2**30 < 2, "STOP: previous model still occupies GPU."
start=time.monotonic()
torch.cuda.reset_peak_memory_stats()
pipe=QwenImage21Pipeline.from_pretrained(MODEL_ID,revision=REVISION,dtype=torch.bfloat16,token=False).to("cuda")
components={}
for name,module in pipe.components.items():
    if isinstance(module,torch.nn.Module):
        dtype_count=collections.Counter()
        device_count=collections.Counter()
        for p in module.parameters():
            dtype_count[str(p.dtype)]+=p.numel()
            device_count[str(p.device)]+=p.numel()
        components[name]={"parameters_by_dtype":dict(dtype_count),"parameters_by_device":dict(device_count),"quantized":bool(getattr(module,"is_quantized",False))}
        assert set(dtype_count)=={"torch.bfloat16"}, f"STOP: unexpected dtype in {name}"
        assert set(device_count)=={"cuda:0"}, f"STOP: off-device weights in {name}"
        assert not components[name]["quantized"], f"STOP: quantization in {name}"
print("IMAGE_COMPONENTS:",json.dumps(components,indent=2))
image_result={"model":MODEL_ID,"revision":REVISION,"diffusers_version":diffusers.__version__,"diffusers_commit":installed_diffusers["source"]["commit"],"diffusers_expected_commit":DIFFUSERS_COMMIT,"diffusers_source":installed_diffusers["source"],"gpu":torch.cuda.get_device_name(0),"components":components,"load_seconds":round(time.monotonic()-start,2),"tests":[]}
steps_seen=[]
def validate_latents(pipeline, step, timestep, callback_kwargs):
    if step%5==0 or step==39:
        assert torch.isfinite(callback_kwargs["latents"]).all().item(), "STOP: non-finite latents"
    steps_seen.append(int(step))
    return callback_kwargs
prompt='A photorealistic red ceramic teapot on a wooden table beside a small green plant, soft morning window light, crisp detail, a white card clearly labeled "A100 BF16 TEST".'
start=time.monotonic()
image=pipe(prompt=prompt,width=2048,height=2048,num_inference_steps=40,true_cfg_scale=1.0,generator=torch.Generator("cuda").manual_seed(42),callback_on_step_end=validate_latents,callback_on_step_end_tensor_inputs=["latents"]).images[0]
torch.cuda.synchronize()
path=REPORT_DIR/"qwen_image_2_1_bf16_2048.png"
image.save(path)
pixels=np.asarray(image)
assert image.size==(2048,2048)
assert np.isfinite(pixels).all() and float(pixels.std())>2, "STOP: invalid/constant image"
image_result["tests"].append({"task":"text_to_image","prompt":prompt,"width":image.width,"height":image.height,"mode":image.mode,"requested_steps":40,"actual_steps":len(steps_seen),"seed":42,"seconds":round(time.monotonic()-start,2),"pixel_std":float(pixels.std()),"file":path.name,"sha256":hashlib.sha256(path.read_bytes()).hexdigest()})
image_result["peak_allocated_gib"]=round(torch.cuda.max_memory_allocated()/2**30,3)
image_result["peak_reserved_gib"]=round(torch.cuda.max_memory_reserved()/2**30,3)
image_result["status"]="INFERENCE_COMPLETED_PENDING_VISUAL_AND_INDEPENDENT_REVIEW"
(REPORT_DIR/"qwen_image_2_1_bf16.json").write_text(json.dumps(image_result,ensure_ascii=False,indent=2))
print("QWEN_IMAGE_BF16_RESULT:",json.dumps(image_result,ensure_ascii=False,indent=2))
display(image)
del pipe,module,p
gc.collect()
torch.cuda.empty_cache()
print("IMAGE_WEIGHTS_UNLOADED_GPU_ALLOCATED_GIB",round(torch.cuda.memory_allocated()/2**30,3))



Download the ZIP before shutdown. If download fails, use Colab Files to download JSON/PNG manually. Do not leave paid GPU running while troubleshooting export. A PNG filename/hash in JSON does not prove the file was exported.


In [ ]:
# Export only the selected run's explicit file allowlist.
from google.colab import files
versions = {package: observed_distribution(package) for package in ["torch", "transformers", "accelerate", "huggingface_hub", "safetensors", "diffusers", "numpy", "Pillow"]}
(REPORT_DIR / "runtime_versions.json").write_text(json.dumps(versions, indent=2))
archive, manifest = export_allowlisted(REPORT_DIR, MODEL_PROFILE)
print("EVIDENCE_EXPORT_FILES", sorted(manifest))
print("EXPORT_SHA256", hashlib.sha256(archive.read_bytes()).hexdigest())
print("ZIP contains exactly manifest-listed payload files plus sha256_manifest.json.")
print("Next: disconnect and delete this paid runtime; verify zero active sessions.")
files.download(str(archive))


In [ ]:
shutdown_now(shutdown_state, runtime.unassign)
